In [3]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

RANDOM_STATE = 42
TARGET = "Churned"
DATA_PATH = Path("ecommerce_customer_churn_dataset.csv")

print("Python:", sys.version.split()[0])
print("Environment:", sys.executable)
print("scikit-learn:", sklearn.__version__)
print("Dataset found:", DATA_PATH.exists())

Python: 3.13.9
Environment: c:\Users\User\anaconda3\python.exe
scikit-learn: 1.7.2
Dataset found: True


In [4]:
from sklearn.model_selection import train_test_split, StratifiedKFold

df = pd.read_csv(DATA_PATH)

X = df.drop(columns=TARGET)
y = df[TARGET].astype(int)

Xtr, Xte, ytr, yte = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

cv5 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Training features:", Xtr.shape)
print("Test features:", Xte.shape)
print(f"Training churn rate: {ytr.mean():.1%}")
print("Overlapping row indices:", len(Xtr.index.intersection(Xte.index)))

Training features: (40000, 24)
Test features: (10000, 24)
Training churn rate: 28.9%
Overlapping row indices: 0


In [5]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline


class CustomerFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # Correct invalid values.
        X.loc[(X["Age"] < 18) | (X["Age"] > 100), "Age"] = np.nan
        X.loc[X["Total_Purchases"] < 0, "Total_Purchases"] = np.nan
        X["Total_Purchases"] = X["Total_Purchases"].round()

        for column in ["Cart_Abandonment_Rate", "Discount_Usage_Rate"]:
            X[column] = X[column].clip(upper=100)

        # Create the five engineered features.
        tenure = X["Membership_Years"].where(
            X["Membership_Years"] > 0, np.nan
        )

        X["Purchase_Frequency"] = X["Total_Purchases"] / tenure
        X["Estimated_Spend"] = (
            X["Total_Purchases"] * X["Average_Order_Value"]
        )
        X["Engagement_Index"] = (
            X["Login_Frequency"]
            * X["Session_Duration_Avg"]
            * X["Pages_Per_Session"]
        )
        X["Service_Calls_per_Year"] = (
            X["Customer_Service_Calls"] / tenure
        )
        X["Recency_to_Tenure"] = (
            X["Days_Since_Last_Purchase"] / (tenure * 365)
        )

        return X.drop(columns="City")


class PercentileCapper(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        values = np.asarray(X, dtype=float)
        self.lower_bounds_ = np.nanquantile(
            values, self.lower, axis=0
        )
        self.upper_bounds_ = np.nanquantile(
            values, self.upper, axis=0
        )
        return self

    def transform(self, X):
        return np.clip(
            np.asarray(X, dtype=float),
            self.lower_bounds_,
            self.upper_bounds_
        )


NUMERIC_FEATURES = [
    "Age", "Membership_Years", "Login_Frequency",
    "Session_Duration_Avg", "Pages_Per_Session",
    "Cart_Abandonment_Rate", "Wishlist_Items", "Total_Purchases",
    "Average_Order_Value", "Days_Since_Last_Purchase",
    "Discount_Usage_Rate", "Returns_Rate", "Email_Open_Rate",
    "Customer_Service_Calls", "Product_Reviews_Written",
    "Social_Media_Engagement_Score", "Mobile_App_Usage",
    "Payment_Method_Diversity", "Lifetime_Value", "Credit_Balance",
    "Purchase_Frequency", "Estimated_Spend", "Engagement_Index",
    "Service_Calls_per_Year", "Recency_to_Tenure"
]

CATEGORICAL_FEATURES = ["Gender", "Country", "Signup_Quarter"]


def make_preparation():
    numeric_steps = Pipeline([
        ("cap", PercentileCapper()),
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ])

    categorical_steps = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(
            drop="first",
            handle_unknown="ignore"
        ))
    ])

    columns = ColumnTransformer([
        ("numeric", numeric_steps, NUMERIC_FEATURES),
        ("categorical", categorical_steps, CATEGORICAL_FEATURES)
    ])

    return Pipeline([
        ("features", CustomerFeatures()),
        ("columns", columns)
    ])


print("Preprocessing ready")

Preprocessing ready


In [6]:
from sklearn.base import clone
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    VotingClassifier
)


def prepared_model(classifier):
    return Pipeline([
        ("preparation", make_preparation()),
        ("classifier", classifier)
    ])


# Confirmed best settings from Chapter 6.
rf_tuned = prepared_model(
    RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_leaf=3,
        min_samples_split=10,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=1
    )
)

hgb_tuned = prepared_model(
    HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=200,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        class_weight="balanced",
        early_stopping=False,
        random_state=RANDOM_STATE
    )
)

voting_ensemble = VotingClassifier(
    estimators=[
        ("rf", clone(rf_tuned)),
        ("hgb", clone(hgb_tuned))
    ],
    voting="soft",
    weights=[1, 1],
    n_jobs=1
)

print("Tuned Random Forest — ready")
print("Tuned Histogram Gradient Boosting — ready")
print("Equal-weight soft voting ensemble — ready")

Tuned Random Forest — ready
Tuned Histogram Gradient Boosting — ready
Equal-weight soft voting ensemble — ready


In [11]:
from sklearn.model_selection import cross_validate

chapter7_models = {
    "Tuned Random Forest": rf_tuned,
    "Tuned Histogram Gradient Boosting": hgb_tuned,
    "Soft Voting Ensemble": voting_ensemble
}

scoring = {
    "roc_auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

comparison_rows = []
chapter7_fold_results = {}

for name, model in chapter7_models.items():
    print(f"Running {name}...", flush=True)

    scores = cross_validate(
        model,
        Xtr,
        ytr,
        cv=cv5,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1,
        error_score="raise"
    )

    chapter7_fold_results[name] = scores

    comparison_rows.append({
        "Model": name,
        "Train ROC-AUC": scores["train_roc_auc"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean(),
        "CV ROC-AUC SD": scores["test_roc_auc"].std(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1": scores["test_f1"].mean(),
        "Mean fit time (s)": scores["fit_time"].mean()
    })

    print(f"Completed {name}", flush=True)

chapter7_results = pd.DataFrame(comparison_rows).set_index("Model")
display(chapter7_results.round(4))

Running Tuned Random Forest...
Completed Tuned Random Forest
Running Tuned Histogram Gradient Boosting...
Completed Tuned Histogram Gradient Boosting
Running Soft Voting Ensemble...
Completed Soft Voting Ensemble


,Train ROC-AUC,CV ROC-AUC,CV ROC-AUC SD,CV Precision,CV Recall,CV F1,Mean fit time (s)
Model,,,,,,,
Tuned Random Forest,0.9973,0.9217,0.0043,0.8738,0.8080,0.8396,7.9611
Tuned Histogram Gradient Boosting,0.9477,0.9265,0.0034,0.8252,0.8535,0.8391,0.6780
Soft Voting Ensemble,0.9857,0.9251,0.0041,0.8505,0.8406,0.8455,12.2083


In [15]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

stacking_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

stacking_ensemble = StackingClassifier(
    estimators=[
        ("rf", clone(rf_tuned)),
        ("hgb", clone(hgb_tuned))
    ],
    final_estimator=LogisticRegression(
        max_iter=3000,
        random_state=RANDOM_STATE
    ),
    stack_method="predict_proba",
    cv=stacking_cv,
    passthrough=False,
    n_jobs=1
)

print("Stacking ensemble — ready")

Stacking ensemble — ready


In [12]:
from sklearn.model_selection import cross_validate

chapter7_models = {
    "Tuned Random Forest": rf_tuned,
    "Tuned Histogram Gradient Boosting": hgb_tuned,
    "Stacking Ensemble": stacking_ensemble
}

scoring = {
    "roc_auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

comparison_rows = []
chapter7_fold_results = {}

for name, model in chapter7_models.items():
    print(f"Running {name}...", flush=True)

    scores = cross_validate(
        model,
        Xtr,
        ytr,
        cv=cv5,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1,
        error_score="raise"
    )

    chapter7_fold_results[name] = scores

    comparison_rows.append({
        "Model": name,
        "Train ROC-AUC": scores["train_roc_auc"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean(),
        "CV ROC-AUC SD": scores["test_roc_auc"].std(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1": scores["test_f1"].mean(),
        "Mean fit time (s)": scores["fit_time"].mean()
    })

    print(f"Completed {name}", flush=True)

stacking_results = pd.DataFrame(comparison_rows).set_index("Model")
display(stacking_results.round(4))

Running Tuned Random Forest...
Completed Tuned Random Forest
Running Tuned Histogram Gradient Boosting...
Completed Tuned Histogram Gradient Boosting
Running Stacking Ensemble...
Completed Stacking Ensemble


,Train ROC-AUC,CV ROC-AUC,CV ROC-AUC SD,CV Precision,CV Recall,CV F1,Mean fit time (s)
Model,,,,,,,
Tuned Random Forest,0.9973,0.9217,0.0043,0.8738,0.8080,0.8396,15.3251
Tuned Histogram Gradient Boosting,0.9477,0.9265,0.0034,0.8252,0.8535,0.8391,1.0041
Stacking Ensemble,0.9766,0.9259,0.0039,0.8736,0.8266,0.8494,44.1690


In [14]:
from sklearn.model_selection import cross_validate

chapter7_models = {
    "Tuned Random Forest": rf_tuned,
    "Tuned Histogram Gradient Boosting": hgb_tuned,
    "Soft Voting Ensemble": voting_ensemble,
    "Stacking Ensemble": stacking_ensemble
}

scoring = {
    "roc_auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

comparison_rows = []
chapter7_fold_results = {}

for name, model in chapter7_models.items():
    print(f"Running {name}...", flush=True)

    scores = cross_validate(
        model,
        Xtr,
        ytr,
        cv=cv5,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1,
        error_score="raise"
    )

    chapter7_fold_results[name] = scores

    comparison_rows.append({
        "Model": name,
        "Train ROC-AUC": scores["train_roc_auc"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean(),
        "CV ROC-AUC SD": scores["test_roc_auc"].std(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1": scores["test_f1"].mean(),
        "Mean fit time (s)": scores["fit_time"].mean()
    })

    print(f"Completed {name}", flush=True)

chapter7_results = pd.DataFrame(comparison_rows).set_index("Model")

display(chapter7_results.round(4))

Running Tuned Random Forest...
Completed Tuned Random Forest
Running Tuned Histogram Gradient Boosting...
Completed Tuned Histogram Gradient Boosting
Running Soft Voting Ensemble...
Completed Soft Voting Ensemble
Running Stacking Ensemble...
Completed Stacking Ensemble


,Train ROC-AUC,CV ROC-AUC,CV ROC-AUC SD,CV Precision,CV Recall,CV F1,Mean fit time (s)
Model,,,,,,,
Tuned Random Forest,0.9973,0.9217,0.0043,0.8738,0.8080,0.8396,8.0012
Tuned Histogram Gradient Boosting,0.9477,0.9265,0.0034,0.8252,0.8535,0.8391,0.6823
Soft Voting Ensemble,0.9857,0.9251,0.0041,0.8505,0.8406,0.8455,9.1374
Stacking Ensemble,0.9766,0.9259,0.0039,0.8736,0.8266,0.8494,45.1990
